In [ ]:
from itertools import combinations
from pathlib import Path

import mlflow
import pandas as pd

from countcv.effcv.plotting_pipeline import (
	MetricCol,
	ParamCol,
	aggregate_experiments_by_seed,
	get_runs_from_csv_spec,
	init_plot_env,
	plot_epoch_metrics_across_runs,
	plot_scatter,
)

# How to plot mlflow results
1. Init
	- Adapt absolute Path to local mlflow folder containing mlflow runs copied from clusters or local runs
	- set OUT_DIR
	- always run init_plot_env(mlrun_abs_path) before plotting

In [ ]:
# TODO !!! must be absolute -> adapt path !!!
mlrun_abs_path = Path("/home/hartwigt/Documents/rotationally-invariant-cnns/mlruns")
OUT_DIR = Path("../plots")
init_plot_env(mlrun_abs_path, out_dir=OUT_DIR)

2. Obtain mlflow runs by ..

- 2.1. using csv file -> samples scatter plots
- 2.2. using mlflow filter options -> sample epoch plot

In [ ]:
# 2.1 using csv spec like the file "config/final_experiments.csv" with columns "experiment_name","run_id"
runs = get_runs_from_csv_spec(Path("../config/final_experiments.csv"))
print(runs.columns)

In [ ]:
# Optional: aggregate experimental results that only differ in random seed
runs = aggregate_experiments_by_seed(runs, verbose=False)
# df_aggregated.head()

In [ ]:
# sample scatter plot
plot_scatter(
	runs,
	x_col=MetricCol.MAE,
	y_col=MetricCol.ENERGY_JOULE,
	dataset_name=None,
	hue_param=ParamCol.TRAIN_SIZE,
	symbol_param=ParamCol.MODEL_VERSION,
	show=True,
	out_dir=OUT_DIR,
)

plot_scatter(
	runs,
	x_col=ParamCol.TRAIN_SIZE,
	y_col=MetricCol.ENERGY_EFFICIENCY,
	# dataset_name=Datasets.SYNTH_CELLS,
	hue_param=ParamCol.TRAIN_SIZE,
	symbol_param=None,
	show=True,
	out_dir=OUT_DIR,
)

In [ ]:
# 2.2. filter runs using mlflow filter options
# get max. 5 runs for params.approach='densitymap'
runs_density = mlflow.search_runs(
	max_results=5,
	filter_string=f"{ParamCol.APPROACH.value}='densitymap'",
	output_format="pandas",
	search_all_experiments=True,
)
assert isinstance(runs_density, pd.DataFrame)
run_ids = runs_density["run_id"].to_list()
# sample mae epoch plot
plot_epoch_metrics_across_runs(run_ids=run_ids, metric=MetricCol.ENERGY_EFFICIENCY, show=True, out_dir=OUT_DIR)

In [ ]:
# plot all combinations of metrics as scatter (creates lots of plots, takes some time)
for x, y in combinations(list(MetricCol), 2):
	if runs[x].nunique() == 1 or runs[y].nunique() == 1:
		continue
	plot_scatter(runs, x_col=x, y_col=y, symbol_param="params.dataset_name", out_dir=OUT_DIR)
	for dataset in runs["params.dataset_name"].unique():
		plot_scatter(runs, x_col=x, y_col=y, dataset_name=dataset, symbol_param="params.model_name", out_dir=OUT_DIR)
		plot_scatter(runs, x_col=x, y_col=y, dataset_name=dataset, symbol_param="params.version", out_dir=OUT_DIR)